In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Estimator wrapper demo

Quick sanity check for the `Estimator` that combines `AbstractGraphTransformer`, an optional manifold step, and a downstream scikit estimator.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from abstractgraph.utils import plot_dataset_method_bars, plot_pareto
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:
import os
os.environ["KMP_WARNINGS"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
# Use shared plotting helper
try:
    from utils import plot_embedding_2d
except ImportError:
    import sys
    from pathlib import Path
    root = Path.cwd()
    for parent in [root, *root.parents]:
        if (parent / "utils.py").exists():
            sys.path.append(str(parent))
            break
    from utils import plot_embedding_2d


In [ ]:
def _compute_Z(model, graphs, embs=None):
    try:
        import warnings
        import umap.umap_ as umap
        if embs is None:
            embs = model.transform(graphs)
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r".*n_jobs value .* overridden .* by setting random_state.*",
                category=UserWarning,
                module="umap.umap_",
            )
            Z = umap.UMAP(n_components=2, random_state=42, init="random").fit_transform(embs)
        return Z
    except Exception:
        return None

def plot(model, graphs, targets, graphs_te=None, targets_te=None):
    import matplotlib.pyplot as _plt
    Z_all = _compute_Z(model, graphs)
    # Full dataset
    fig, axes = _plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix='All data', mode='scatter', alpha=0.75, ax=axes[0], show=False, Z=Z_all)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix='All data', mode='knn_class_union', k=5, alpha=0.75, z=1, ax=axes[1], show=False, show_instances=True, Z=Z_all)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix='All data', mode='knn_class_union', k=11, alpha=0.75, z=1, ax=axes[2], show=False, show_instances=True, Z=Z_all)
    _plt.show()
    if graphs_te is None or targets_te is None:
        return
    # Test split
    Z_te = _compute_Z(model, graphs_te)
    fig, axes = _plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    _ = plot_embedding_2d(model, graphs_te, targets_te, title_prefix='Test', mode='scatter', alpha=0.75, ax=axes[0], show=False, Z=Z_te)
    _ = plot_embedding_2d(model, graphs_te, targets_te, title_prefix='Test', mode='knn_class_union', k=5, alpha=0.75, z=1, ax=axes[1], show=False, show_instances=True, Z=Z_te)
    _ = plot_embedding_2d(model, graphs_te, targets_te, title_prefix='Test', mode='knn_class_union', k=11, alpha=0.75, z=1, ax=axes[2], show=False, show_instances=True, Z=Z_te)
    _plt.show()


In [ ]:
from abstractgraph_graphicalizer.chem import PubChemAssayLoader

loader = PubChemAssayLoader(on_error="skip")
assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_ids = assay_ids[:4]
datasets = []
for assay_id in assay_ids:
    original_graphs, original_targets = loader.load(assay_id, limit_active=1500, limit_inactive=1500)
    print(f'AID{assay_id}  #graphs: {len(original_graphs)}')
    datasets.append((assay_id, original_graphs, original_targets))

---

In [ ]:
from abstractgraph.operators import *

decomposition_functions = []

df = neighborhood(radius=(0,2))
decomposition_functions.append(('neighborhood',df))

cycle_and_tree = (cycle(), compose(neighborhood(radius=1), tree()))
df = add( *cycle_and_tree, compose_product(binary_combination(distance=0), *cycle_and_tree))
decomposition_functions.append(('cycle',df))

df = graphlet(radius=3, number_of_nodes=(4,6))
#decomposition_functions.append(('graphlet',df))

df = add(neighborhood(radius=(0,2)), compose(combination(number_of_elements=2,distance=(1,3)), neighborhood(radius=1)))
#decomposition_functions.append(('pairs',df))

In [ ]:
%%time
from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph_ml.estimators import GraphEstimator
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from matplotlib.colors import ListedColormap

for assay_id, graphs, targets in datasets:
    print(f"Processing assay {assay_id} (n={len(graphs)})")
    for decomposition_name, decomposition_fn in decomposition_functions:
        print(f"  Decomposition: {decomposition_name}")
        transformer = AbstractGraphTransformer(
            nbits=11,
            decomposition_function=decomposition_fn,
            return_dense=True,
            n_jobs=-1,  # avoid multiprocessing pickling issues in notebooks
        )

        graph_estimator = GraphEstimator(
            transformer=transformer,
            estimator=None,
            manifold=None,
        )
        graph_estimator.fit(graphs, targets)
        plot(graph_estimator, graphs, targets)


---

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph_ml.estimators import GraphEstimator

# Interactive viewer: select dataset/decomposition and adjust k for knn overlays
assay_options = [(f"{i}: {aid}", i) for i, (aid, _, _) in enumerate(datasets)]
decomp_options = [(name, idx) for idx, (name, _) in enumerate(decomposition_functions)]

assay_dd = widgets.Dropdown(options=assay_options, description='Assay:')
decomp_dd = widgets.Dropdown(options=decomp_options, description='Decomp:')
k_slider = widgets.IntSlider(value=5, min=2, max=15, step=1, description='k:', continuous_update=False)

out = widgets.Output()
state = {'key': None, 'Z': None, 'graphs': None, 'labels': None, 'est': None}


def _compute_embedding(assay_idx, decomp_idx):
    assay_id, graphs, targets = datasets[assay_idx]
    decomp_name, decomp_fn = decomposition_functions[decomp_idx]
    transformer = AbstractGraphTransformer(
        nbits=11,
        decomposition_function=decomp_fn,
        return_dense=True,
        n_jobs=-1,
    )
    est = GraphEstimator(
        transformer=transformer,
        estimator=None,
        manifold=None,
    )
    est.fit(graphs, targets)
    embs = est.transform(graphs)
    try:
        Z = _compute_Z(est, graphs, embs=embs)
    except TypeError:
        # Backward compatibility if _compute_Z does not accept embs yet
        Z = _compute_Z(est, graphs)
    if Z is None:
        Z = embs
    return est, graphs, targets, Z, f"AID {assay_id} — {decomp_name}"


def _update(_=None):
    a_idx = assay_dd.value
    d_idx = decomp_dd.value
    k = k_slider.value
    key = (a_idx, d_idx)
    if state['key'] != key:
        est, graphs, labels, Z, title = _compute_embedding(a_idx, d_idx)
        state.update({'key': key, 'est': est, 'graphs': graphs, 'labels': labels, 'Z': Z, 'title': title})
    est = state['est']
    graphs = state['graphs']
    labels = state['labels']
    Z = state['Z']
    title = state['title']
    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
        plot_embedding_2d(est, graphs, labels, title_prefix=title, mode='scatter', alpha=0.75, ax=axes[0], show=False, Z=Z)
        plot_embedding_2d(est, graphs, labels, title_prefix=title, mode='knn_class_union', k=k, alpha=0.75, z=1, ax=axes[1], show=False, show_instances=True, Z=Z)
        plt.show()


assay_dd.observe(_update, names='value')
decomp_dd.observe(_update, names='value')
k_slider.observe(_update, names='value')

controls = widgets.HBox([assay_dd, decomp_dd, k_slider])
display(controls, out)
_update()
